<a href="https://colab.research.google.com/github/ahmedamraziz/Personalized-Chatbot-with-Dynamic-Responses/blob/main/labor_law_chatbot_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setup & Install Libraries
1.1 Mount Google Drive

In [ ]:
# ============================================
# MOUNT GOOGLE DRIVE (COLAB)
# ============================================

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


1.2 Install Required Libraries

In [ ]:
# ============================================
# INSTALL REQUIRED LIBRARIES
# ============================================

!pip install -q \
pypdf \
sentence-transformers \
transformers \
numpy \
scikit-learn \
chromadb \
bitsandbytes \
accelerate \
streamlit \
pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/1

# 2. Import Libraries
2.1 PDF Processing

In [ ]:
# ============================================
# PDF PROCESSING
# ============================================

from pypdf import PdfReader

2.2 NLP & Utilities

In [ ]:
# ============================================
# NLP & UTILITIES
# ============================================

import re
import os
import numpy as np

from collections import Counter
from nltk.tokenize import sent_tokenize

2.3 Embedding & Reranking Models

In [ ]:
# ============================================
# EMBEDDING & RERANKING
# ============================================

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from sklearn.metrics.pairwise import cosine_similarity

2.4 ChromaDB

In [ ]:
# ============================================
# CHROMADB
# ============================================

import chromadb
from chromadb.utils import embedding_functions

2.5 LLM (Mistral)

In [ ]:
# ============================================
# LLM
# ============================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# 3. Read PDF File
3.1 PDF Reader Function

In [ ]:
# ============================================
# READ PDF
# ============================================

def read_pdf(file_path):

    reader = PdfReader(file_path)

    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    return text

3.2 Load PDF

In [ ]:
# ============================================
# LOAD PDF FILE
# ============================================

file_path = "/content/Labor-law (1).pdf"

text = read_pdf(file_path)

text = text.strip()

print(text[:1000])

Preamble 
In the Name of the People 
The President of the Republic 
The House of Representatives has enacted the following law, which we hereby 
issue: 
 
 
Article 1 – Promulgation: 
The provisions of this law and the accompanying law shall govern employment matters. 
They shall also apply, in the absence of specific provisions in individual employment contracts or 
collective labor agreements, to foreign workers within the Arab Republic of Egypt. 
Unless otherwise specifically provided, the provisions of this law and the accompanying law shall 
not apply to the following categories: 
• Employees of government bodies, including local administration units and public 
authorities. 
• Domestic workers and those in similar categories. 
 
Article 2 – Promulgation: 
The Training and Qualification Fund, established under Labor Law No. 12 of 2003, shall retain its 
legal public personality, report to the Minister responsible for labor affairs, and exercise its 
powers as regulated by the acco

# 4. Split PDF into Articles
4.1 Split Articles Function

In [ ]:
# ============================================
# SPLIT INTO ARTICLES
# ============================================

def split_by_articles(text):

    if text is None:
        return []

    if isinstance(text, list):

        if len(text) > 0 and isinstance(text[0], dict):
            text = "\n".join(x.get("text", "") for x in text)

        else:
            text = "\n".join(map(str, text))

    elif isinstance(text, dict):
        text = text.get("text", "")

    text = str(text)

    text = text.replace("\r", "\n")

    text = re.sub(r"\n+", "\n", text).strip()

    pattern = r"(?m)^(Article\s+\d+\s*(?:[–\-:][^\n]*)?:?)"

    parts = re.split(pattern, text)

    articles = []

    if len(parts) > 0 and not re.match(r"Article\s+\d+", parts[0]):
        parts = parts[1:]

    for i in range(0, len(parts) - 1, 2):

        title = parts[i].strip()

        content = parts[i + 1].strip()

        if not title or len(content.split()) < 10:
            continue

        articles.append({
            "article": title,
            "text": content
        })

    return articles

4.2 Run Article Splitting

In [ ]:
# ============================================
# RUN ARTICLE SPLITTING
# ============================================

articles = split_by_articles(text)

print("Total Articles:", len(articles))

if len(articles) > 0:

    print("\nFIRST ARTICLE:\n")

    print("TITLE:", articles[0]["article"])

    print("\nTEXT:\n")

    print(articles[0]["text"][:500])

else:
    print("No articles found")

Total Articles: 311

FIRST ARTICLE:

TITLE: Article 1 – Promulgation:

TEXT:

The provisions of this law and the accompanying law shall govern employment matters. 
They shall also apply, in the absence of specific provisions in individual employment contracts or 
collective labor agreements, to foreign workers within the Arab Republic of Egypt. 
Unless otherwise specifically provided, the provisions of this law and the accompanying law shall 
not apply to the following categories: 
• Employees of government bodies, including local administration units and public 
authorit


# 5. Parse & Clean Articles
5.1 Parse Article Title

In [ ]:
# ============================================
# PARSE ARTICLE TITLE
# ============================================

def parse_article_title(title):

    match = re.match(
        r"Article\s+(\d+)\s*(?:[–\-]\s*(.*))?:?",
        title,
        re.IGNORECASE
    )

    if not match:
        return None, None

    number = int(match.group(1))

    subtitle = (
        match.group(2)
        if match.group(2)
        else "General"
    )

    return number, subtitle.strip()

5.2 Clean Articles

In [ ]:
# ============================================
# CLEAN ARTICLES
# ============================================

def clean_articles(articles):

    cleaned = []

    for a in articles:

        num, title = parse_article_title(a["article"])

        if num is None:
            continue

        cleaned.append({
            "id": num,
            "title": title,
            "text": a["text"],
            "raw": a["article"]
        })

    return cleaned

5.3 Run Cleaning

In [ ]:
# ============================================
# RUN CLEANING
# ============================================

articles = clean_articles(articles)

print(articles[0])

{'id': 1, 'title': 'Promulgation:', 'text': 'The provisions of this law and the accompanying law shall govern employment matters. \nThey shall also apply, in the absence of specific provisions in individual employment contracts or \ncollective labor agreements, to foreign workers within the Arab Republic of Egypt. \nUnless otherwise specifically provided, the provisions of this law and the accompanying law shall \nnot apply to the following categories: \n• Employees of government bodies, including local administration units and public \nauthorities. \n• Domestic workers and those in similar categories.', 'raw': 'Article 1 – Promulgation:'}


# 6. Standardize Article Titles
6.1 Title Formatter

In [ ]:
# ============================================
# FIX ARTICLE TITLE FORMAT
# ============================================

def fix_title(article_id, raw_title):

    raw_title = raw_title.replace("Article", "").strip()

    if "–" in raw_title:

        parts = raw_title.split("–", 1)

        num = parts[0].strip()

        title = parts[1].strip()

    elif ":" in raw_title:

        parts = raw_title.split(":", 1)

        num = parts[0].strip()

        title = parts[1].strip()

    else:

        num = str(article_id)

        title = "General"

    return f"Article {num} - {title}"

6.2 Apply Formatting

In [ ]:
# ============================================
# APPLY TITLE FORMAT
# ============================================

for a in articles:

    a["title"] = fix_title(
        a["id"],
        a["raw"]
    )

print(articles[0]["title"])
print(articles[13]["title"])

Article 1 - Promulgation:
Article 1 - 


# 7. Load Embedding & Reranking Models
7.1 Embedding Model

In [ ]:
# ============================================
# LOAD EMBEDDING MODEL
# ============================================

embedder = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

7.2 Cross Encoder Reranker

In [ ]:
# ============================================
# LOAD RERANKER
# ============================================

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

# 8. Generate Embeddings
8.1 Embedding Function

In [ ]:
# ============================================
# CREATE EMBEDDINGS
# ============================================

def prepare_embeddings(articles):

    texts = [a["text"] for a in articles]

    embeddings = embedder.encode(
        texts,
        normalize_embeddings=True
    )

    return np.array(embeddings)

8.2 Build Embeddings

In [ ]:
# ============================================
# BUILD EMBEDDINGS
# ============================================

embeddings = prepare_embeddings(articles)

print(embeddings.shape)

(311, 768)


# 9. Hybrid Retrieval System
9.1 Keyword Score

In [ ]:
# ============================================
# KEYWORD SCORE
# ============================================

def keyword_score(query, text):

    query_tokens = re.findall(
        r"\w+",
        query.lower()
    )

    text_tokens = re.findall(
        r"\w+",
        text.lower()
    )

    if not text_tokens:
        return 0

    text_counter = Counter(text_tokens)

    score = 0

    for token in query_tokens:
        score += text_counter.get(token, 0)

    return score / (len(text_tokens) + 1e-9)

9.2 Hybrid Retrieval + Reranking

In [ ]:
# ============================================
# HYBRID RETRIEVAL
# ============================================

def retrieve(
    query,
    articles,
    embeddings,
    top_k=5,
    alpha=0.75,
    rerank_top_k=10
):

    # ====================================
    # Semantic Search
    # ====================================

    q_emb = embedder.encode(
        [query],
        normalize_embeddings=True
    )

    semantic_scores = cosine_similarity(
        embeddings,
        q_emb
    ).flatten()

    semantic_scores = (
        semantic_scores - semantic_scores.min()
    ) / (
        semantic_scores.max()
        - semantic_scores.min()
        + 1e-9
    )

    # ====================================
    # Keyword Search
    # ====================================

    keyword_scores = np.array([
        keyword_score(query, a["text"])
        for a in articles
    ])

    if keyword_scores.max() > 0:

        keyword_scores = (
            keyword_scores /
            (keyword_scores.max() + 1e-9)
        )

    # ====================================
    # Hybrid Fusion
    # ====================================

    hybrid_scores = (
        alpha * semantic_scores
        + (1 - alpha) * keyword_scores
    )

    # ====================================
    # Top Candidates
    # ====================================

    top_idx = np.argsort(
        hybrid_scores
    )[::-1][:rerank_top_k]

    candidates = [
        articles[i]
        for i in top_idx
    ]

    # ====================================
    # Reranking
    # ====================================

    pairs = [
        (query, c["text"])
        for c in candidates
    ]

    rerank_scores = reranker.predict(pairs)

    reranked = sorted(
        zip(candidates, rerank_scores),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    # ====================================
    # Final Results
    # ====================================

    results = []

    for doc, score in reranked:

        results.append({
            "id": doc["id"],
            "title": doc["title"],
            "text": doc["text"],
            "score": float(score)
        })

    return results

9.3 Test Retrieval

In [ ]:
# ============================================
# TEST RETRIEVAL
# ============================================

query = "sick leave"

results = retrieve(
    query,
    articles,
    embeddings,
    top_k=5
)

for r in results:

    print(r["title"])

    print(r["text"][:300])

    print("Score:", r["score"])

    print("-" * 50)

NameError: name 'retrieve' is not defined

# 10. ChromaDB Vector Database
10.1 Create Chroma Client

In [ ]:
# ============================================
# CREATE CHROMA CLIENT
# ============================================

client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/chatbot_new "
)

10.2 Embedding Function

In [ ]:
# ============================================
# CHROMA EMBEDDING FUNCTION
# ============================================

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-base-en-v1.5"
)

10.3 Create Collection

In [ ]:
# ============================================
# CREATE COLLECTION
# ============================================

collection = client.get_or_create_collection(
    name="labor_law",
    embedding_function=embed_fn
)

# 11. Save Articles to ChromaDB
11.1 Prepare Data

In [ ]:
# ============================================
# PREPARE DATA
# ============================================

ids = []
documents = []
metadatas = []

for i, a in enumerate(articles):

    ids.append(str(i))

    documents.append(a["text"])

    metadatas.append({
        "title": a["title"]
    })

11.2 Add to ChromaDB

In [ ]:
# ============================================
# ADD TO CHROMADB
# ============================================

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

# 12. Query ChromaDB
12.1 Get data from ChromDB

In [ ]:
pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentel

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

In [ ]:
client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/chatbot_new "
)

In [ ]:
print(client.list_collections())

[]


In [ ]:
collection = client.get_collection(name="labor_law")

In [ ]:
collection.peek(5)

{'ids': ['0', '1', '2', '3', '4'],
 'embeddings': array([[ 0.02527438,  0.0369247 , -0.02211486, ...,  0.00978677,
         -0.02546797,  0.03257608],
        [ 0.05578815, -0.04181446, -0.01721933, ..., -0.02012032,
         -0.06090721,  0.02091472],
        [ 0.04630249, -0.03443568, -0.01498932, ...,  0.00353116,
         -0.04380153,  0.03176032],
        [ 0.01707505, -0.00580141, -0.01055424, ..., -0.02687238,
         -0.06696147,  0.00511791],
        [ 0.03837891, -0.00840427,  0.05261407, ..., -0.00804846,
         -0.04924224,  0.00294173]]),
 'documents': ['The provisions of this law and the accompanying law shall govern employment matters. \nThey shall also apply, in the absence of specific provisions in individual employment contracts or \ncollective labor agreements, to foreign workers within the Arab Republic of Egypt. \nUnless otherwise specifically provided, the provisions of this law and the accompanying law shall \nnot apply to the following categories: \n• Employe

12.2 Display and rerank Results

In [ ]:
# ============================================
# INSTALL LIBRARIES
# ============================================

!pip install chromadb sentence-transformers scikit-learn

# ============================================
# IMPORTS
# ============================================

import chromadb
import numpy as np
import re

from collections import Counter

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from sklearn.metrics.pairwise import cosine_similarity

# ============================================
# LOAD CHROMADB
# ============================================

client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/chatbot_new "
)

collection = client.get_collection(
    name="labor_law"
)

# ============================================
# LOAD MODELS
# ============================================

embedder = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

# ============================================
# GET ALL DATA FROM CHROMADB
# ============================================

data = collection.get(
    include=["documents", "metadatas"]
)

# ============================================
# CONVERT TO ARTICLES FORMAT
# ============================================

articles = []

for i in range(len(data["documents"])):

    articles.append({

        "id": data["ids"][i],

        "title": data["metadatas"][i]["title"],

        "text": data["documents"][i]

    })

print("Total Articles:", len(articles))

# ============================================
# PREPARE EMBEDDINGS
# ============================================

def prepare_embeddings(articles):

    texts = [
        a["text"]
        for a in articles
    ]

    embeddings = embedder.encode(
        texts,
        normalize_embeddings=True
    )

    return np.array(embeddings)

# ============================================
# KEYWORD SCORE
# ============================================

def keyword_score(query, text):

    query_tokens = re.findall(
        r"\w+",
        query.lower()
    )

    text_tokens = re.findall(
        r"\w+",
        text.lower()
    )

    if not text_tokens:
        return 0

    text_counter = Counter(text_tokens)

    score = 0

    for token in query_tokens:

        score += text_counter.get(token, 0)

    return score / (len(text_tokens) + 1e-9)

# ============================================
# HYBRID RETRIEVAL
# ============================================

def retrieve(
    query,
    articles,
    embeddings,
    top_k=5,
    alpha=0.75,
    rerank_top_k=5
):

    # ====================================
    # Semantic Search
    # ====================================

    q_emb = embedder.encode(
        [query],
        normalize_embeddings=True
    )

    semantic_scores = cosine_similarity(
        embeddings,
        q_emb
    ).flatten()

    semantic_scores = (
        semantic_scores - semantic_scores.min()
    ) / (
        semantic_scores.max()
        - semantic_scores.min()
        + 1e-9
    )

    # ====================================
    # Keyword Search
    # ====================================

    keyword_scores = np.array([

        keyword_score(query, a["text"])

        for a in articles
    ])

    if keyword_scores.max() > 0:

        keyword_scores = (
            keyword_scores /
            (keyword_scores.max() + 1e-9)
        )

    # ====================================
    # Hybrid Fusion
    # ====================================

    hybrid_scores = (
        alpha * semantic_scores
        + (1 - alpha) * keyword_scores
    )

    # ====================================
    # TOP CANDIDATES
    # ====================================

    top_idx = np.argsort(
        hybrid_scores
    )[::-1][:rerank_top_k]

    candidates = [

        articles[i]

        for i in top_idx
    ]

    # ====================================
    # RERANKING
    # ====================================

    pairs = [

        (query, c["text"])

        for c in candidates
    ]

    rerank_scores = reranker.predict(
        pairs
    )

    reranked = sorted(

        zip(candidates, rerank_scores),

        key=lambda x: x[1],

        reverse=True

    )[:top_k]

    # ====================================
    # FINAL RESULTS
    # ====================================

    results = []

    for doc, score in reranked:

        results.append({

            "id": doc["id"],

            "title": doc["title"],

            "text": doc["text"],

            "score": float(score)

        })

    return results

# ============================================
# CREATE EMBEDDINGS
# ============================================

embeddings = prepare_embeddings(
    articles
)

# ============================================
# QUERY
# ============================================

query = "what are sick leave rights"

results = retrieve(
    query=query,
    articles=articles,
    embeddings=embeddings,
    top_k=5
)
# ============================================
# FILTER RESULTS
# ============================================

filtered_results = []

for r in results:

    if r["score"] > 0:

        filtered_results.append(r)

# overwrite
results = filtered_results

# ============================================
# DISPLAY RESULTS
# ============================================

for r in results:

    print(r["title"])

    print("Score:", r["score"])

    print(r["text"][:1000])

    print("-" * 70)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Total Articles: 311
Article 131 - 
Score: 4.840604782104492
If the employee is proven to be ill or injured in a manner that prevents them from performing 
their work, they shall be entitled to sick leave as determined by the competent medical authority. 
During such leave, the employee shall receive wage compensation, the amount and duration of 
which shall be in accordance with the Social Insurance and Pensions Law. 
  
 
 
For employees in industrial establishments governed by Law No. 15 of 2017 (Facilitating 
Industrial Licensing), sick leave is granted on the following basis within each three-year service 
period: 
• Three (3) months with full pay, 
• Followed by six (6) months at 85% of the wage, 
• Then three (3) months at 75% of the wage, 
provided that the competent medical authority deems recovery likely. 
 
The employer may deduct from their wage obligation any compensation paid under the social 
insurance system. 
 
The employee may use accrued annual leave in addition to si

In [ ]:
# QUERY
# ============================================

query = "what are sick leave rights"

results = retrieve(
    query=query,
    articles=articles,
    embeddings=embeddings,
    top_k=5
)
# ============================================
# FILTER RESULTS
# ============================================

filtered_results = []

for r in results:

    if r["score"] > 0:

        filtered_results.append(r)

# overwrite
results = filtered_results

# ============================================
# DISPLAY RESULTS
# ============================================

for r in results:

    print(r["title"])

    print("Score:", r["score"])

    print(r["text"][:1000])

    print("-" * 70)

Article 131 - 
Score: 4.840604782104492
If the employee is proven to be ill or injured in a manner that prevents them from performing 
their work, they shall be entitled to sick leave as determined by the competent medical authority. 
During such leave, the employee shall receive wage compensation, the amount and duration of 
which shall be in accordance with the Social Insurance and Pensions Law. 
  
 
 
For employees in industrial establishments governed by Law No. 15 of 2017 (Facilitating 
Industrial Licensing), sick leave is granted on the following basis within each three-year service 
period: 
• Three (3) months with full pay, 
• Followed by six (6) months at 85% of the wage, 
• Then three (3) months at 75% of the wage, 
provided that the competent medical authority deems recovery likely. 
 
The employer may deduct from their wage obligation any compensation paid under the social 
insurance system. 
 
The employee may use accrued annual leave in addition to sick leave, or may req

In [ ]:
# QUERY
# ============================================

query = "what are annual leave rights"

results = retrieve(
    query=query,
    articles=articles,
    embeddings=embeddings,
    top_k=5
)
# ============================================
# FILTER RESULTS
# ============================================

filtered_results = []

for r in results:

    if r["score"] > 1:

        filtered_results.append(r)

# overwrite
results = filtered_results

# ============================================
# DISPLAY RESULTS
# ============================================

for r in results:

    print(r["title"])

    print("Score:", r["score"])

    print(r["text"][:1000])

    print("-" * 70)

Article 125 - 
Score: 5.618236064910889
The employer shall determine the timing of annual leave based on work requirements and 
conditions. Leave may not be interrupted except for compelling reasons required by the interest 
of work. 
 
Employees may not waive their right to annual leave and are obligated to take it on the date and 
for the duration determined and communicated by the employer. If the employee refuses in 
writing to take the leave, they forfeit the right to claim its monetary equivalent. 
 
In all cases, the employee must take at least fifteen (15) days of leave annually, including a 
minimum of six (6) consecutive days. 
 
The employer must settle the employee’s leave balance or its cash equivalent at least once every 
three years. If the employment relationship ends before the employee uses their leave balance, 
they are entitled to receive compensation for the unused portion. 
 
Leave may not be split, carried over, or deferred for children, persons with disabilities

# 13. Download & Load Mistral Model
13.1 Download Model

In [ ]:
# ============================================
# DOWNLOAD MISTRAL MODEL
# ============================================

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    local_dir="/content/drive/MyDrive/mistral",
    local_dir_use_symlinks=False
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

'/content/drive/.shortcut-targets-by-id/1ixsSoXLYEw-D6TpIb2A3SbsNfF52Wdka/mistral'

13.2 Load Quantized Model

In [ ]:
!pip install -U bitsandbytes accelerate transformers

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import torch

In [ ]:
# ============================================
# LOAD QUANTIZED MODEL
# ============================================

# Ensure bitsandbytes is updated to the required version
!pip install -U bitsandbytes>=0.46.1

MODEL_NAME = "/content/drive/MyDrive/mistral"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

# 14. Generate Answers
14.1 Generate Answer Function

In [ ]:
def extract_from_article(query, article, model, tokenizer):

    context = f"""
{article['title']}

{article['text']}
"""

    prompt = f"""
You are a legal extraction assistant.

STRICT RULES:
- Use ONLY this article
- Extract ONLY information relevant to the question
- Do NOT use external knowledge
- Do NOT combine information
- If unrelated return ONLY:
NONE

Return bullet points only.

ARTICLE:
{context}

QUESTION:
{query}

ANSWER:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=800,
        temperature=0.0,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    answer = full_output[len(prompt):].strip()

    return answer

In [ ]:
query = "what are sick leave rights?"

final_answers = []

for article in processed_results:

    response = extract_from_article(
        query,
        article,
        model,
        tokenizer
    )

    if response.strip() != "NONE":

        final_answers.append({
            "article": article["title"],
            "answer": response
        })

In [ ]:
for item in final_answers:

    print(item["article"])

    print(item["answer"])

    print("-" * 50)

Article 131 - 
- Employees are entitled to sick leave based on medical proof
- During sick leave, they receive wage compensation according to the Social Insurance and Pensions Law
- For employees in certain industrial establishments, sick leave is granted for a certain period with varying wage percentages
- Employers can deduct social insurance compensation from their wage obligation
- Employees can use accrued annual leave during sick leave or convert sick leave to annual leave if they have enough balance.
--------------------------------------------------
Article 173 - 
- Employer cannot terminate contract due to worker's illness until sick leave and annual leave are exhausted
- Employer must give notice of intention to terminate within 15 days after leave is exhausted
- Worker can recover before notice is issued, preventing termination due to illness.
--------------------------------------------------


In [ ]:
def generate_answer(query, retrieved_docs, model, tokenizer):

    context = "\n\n".join([
        f"{d['title']}\n{d['text']}" for d in retrieved_docs
    ])

    prompt = f"""
You are a professional labor law assistant.
RULES:
- Structured answer
- Bullet points
- No hallucination
CONTEXT:
{context}
QUESTION:
{query}
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in full_output:
        answer = full_output.split("Answer:")[-1].strip()
    else:
        answer = full_output

    return answer

14.2 Generate Example Answer

In [ ]:
query = "what are sick leave rights?"

# results is the dictionary returned by collection.query
# We need to transform it into the format expected by generate_answer

processed_results = []
num_results_to_take = 3
if results and "documents" in results and results["documents"] and \
   "metadatas" in results and results["metadatas"]:
    for i in range(min(num_results_to_take, len(results["documents"][0]))):
        doc = results["documents"][0][i]
        meta = results["metadatas"][0][i]
        processed_results.append({
            "title": meta.get("title", "No Title"), # Use .get() for safety
            "text": doc
        })

answer = generate_answer(query, processed_results, model, tokenizer)

print(answer)


You are a professional labor law assistant.
RULES:
- Structured answer
- Bullet points
- No hallucination
CONTEXT:
Article 131 - 
If the employee is proven to be ill or injured in a manner that prevents them from performing 
their work, they shall be entitled to sick leave as determined by the competent medical authority. 
During such leave, the employee shall receive wage compensation, the amount and duration of 
which shall be in accordance with the Social Insurance and Pensions Law. 
  
 
 
For employees in industrial establishments governed by Law No. 15 of 2017 (Facilitating 
Industrial Licensing), sick leave is granted on the following basis within each three-year service 
period: 
• Three (3) months with full pay, 
• Followed by six (6) months at 85% of the wage, 
• Then three (3) months at 75% of the wage, 
provided that the competent medical authority deems recovery likely. 
 
The employer may deduct from their wage obligation any compensation paid under the social 
insurance

# 15. RAG Evaluation
15.1 Word Faithfulness

In [ ]:
# ============================================
# WORD FAITHFULNESS
# ============================================

def faithfulness_score(
    answer,
    retrieved_docs
):

    context = " ".join([
        d["text"]
        for d in retrieved_docs
    ]).lower()

    answer = answer.lower()

    words = answer.split()

    if len(words) == 0:
        return 0

    match = sum(
        1 for w in words
        if w in context
    )

    return match / len(words)

15.2 Semantic Faithfulness

In [ ]:
# ============================================
# SEMANTIC FAITHFULNESS
# ============================================

def semantic_faithfulness(
    answer,
    retrieved_docs
):

    context = " ".join([
        d["text"]
        for d in retrieved_docs
    ])

    answer_emb = embedder.encode([answer])

    context_emb = embedder.encode([context])

    score = cosine_similarity(
        answer_emb,
        context_emb
    )[0][0]

    return float(score)

15.3 Final RAG Score

In [ ]:
# ============================================
# FINAL RAG SCORE
# ============================================

def rag_score(
    answer,
    retrieved_docs
):

    f1 = faithfulness_score(
        answer,
        retrieved_docs
    )

    f2 = semantic_faithfulness(
        answer,
        retrieved_docs
    )

    final = 0.5 * f1 + 0.5 * f2

    return {
        "faithfulness_word": f1,
        "faithfulness_semantic": f2,
        "final_score": final
    }

# 16. Streamlit UI
16.1 Create app.py

In [ ]:
%%writefile app.py

import streamlit as st
import chromadb
from chromadb.utils import embedding_functions
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# =========================
# Load Model (cached)
# =========================
@st.cache_resource
def load_model():
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        llm_int8_enable_fp32_cpu_offload=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

    return tokenizer, model

tokenizer, model = load_model()

# =========================
# Load Chroma DB
# =========================
@st.cache_resource
def load_chroma():
    client = chromadb.PersistentClient(
        path="/content/drive/MyDrive/chatbot_chroma"
    )

    embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="BAAI/bge-base-en-v1.5"
    )

    collection = client.get_collection(
        name="labor_law",
        embedding_function=embed_fn
    )

    return collection

collection = load_chroma()

# =========================
# LLM
# =========================
def generate_answer(query, retrieved_docs, model, tokenizer, language):

    context = "\n\n".join([
        f"{d['title']}\n{d['text']}" for d in retrieved_docs
    ])

    if language == "Arabic":
        instruction = """
You are a legal assistant.

Rules:
- Respond ONLY in Arabic
- Use only bullet points
- Maximum 5–7 points
- Keep answers simple and short
- Do not add extra explanations
"""
    else:
        instruction = """
You are a legal assistant.

Rules:
- Respond ONLY in English
- Use only bullet points
- Maximum 5–7 points
- Keep answers simple and short
- Do not add extra explanations
"""

    prompt = f"""
{instruction}

Rules:
- If the answer exists in the context, extract it clearly.
- If multiple parts exist, combine them simply.
- If partially missing, say what is available and mark missing parts.
- Never use external knowledge.
- Never change the topic..

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in full_output:
        answer = full_output.split("Answer:")[-1].strip()
    else:
        answer = full_output

    return answer

# =========================
# UI
# =========================

st.title("🤖 Labor Law Chatbot – Know Your Employee Rights ⚖️")

st.markdown("### 👷 Ask anything about your rights as an employee")

# اختيار اللغة
language = st.selectbox("Choose language / اختر اللغة", ["Arabic", "English"])

query = st.text_input("Ask your question / اكتب سؤالك:")

if st.button("Ask / اسأل"):
    if query:

        results = collection.query(
            query_texts=[query],
            n_results=3
        )

        docs = results["documents"][0]
        metas = results["metadatas"][0]

        retrieved_docs = []
        for doc, meta in zip(docs, metas):
            retrieved_docs.append({
                "title": meta.get("title", "No Title"),
                "text": doc
            })

        response = generate_answer(query, retrieved_docs, model, tokenizer, language)

        st.write("### Answer / الإجابة:")
        st.write(response)

16.2 Run Streamlit

In [ ]:
!cat app.py

In [ ]:
# ============================================
# RUN STREAMLIT
# ============================================

!streamlit run app.py \
--server.port 8501 \
--server.address 0.0.0.0 &

# 17. Public URL with Ngrok
17.1 Start Ngrok

In [ ]:
# ============================================
# NGROK
# ============================================

from pyngrok import ngrok

ngrok.set_auth_token("3CiqFLphdOQ8VjhyJ3PknDMTdDS_7AJs7m7gCsFgje6MMyymr")

public_url = ngrok.connect(8501)

print(public_url)